# 🐍 Agent Memory with Persistent Storage - Microsoft Agent Framework (Python)

## 📋 Scenario Overview

This notebook demonstrates **persistent agent memory** using the Microsoft Agent Framework with file-based storage. We'll build a travel planning agent that remembers user preferences across different conversation sessions using a simple JSON-based memory system.

**Key Concepts:**
- 💾 **Persistent Memory**: User preferences survive across sessions
- 🔍 **Memory Retrieval**: Find relevant past conversations
- 👤 **User Isolation**: Separate memory for different users
- 📊 **Memory Management**: Store, retrieve, and search memories

## ⚙️ Prerequisites

**Required:**
```bash
pip install agent-framework-core azure-identity python-dotenv
```

**Note:** For production use with Mem0 and Azure AI Search, see the Semantic Kernel version

Let's build an agent with long-term memory! 🌟

## Step 1: Install Dependencies and Import Packages

In [ ]:
# Install required packages
# !pip install azure-ai-projects azure-identity python-dotenv

In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    Agent,
    MessageRole,
    ThreadRun,
    RunStatus,
    ToolSet,
    FunctionTool,
    ToolOutput
)
from azure.ai.inference.models import (
    ChatRequestMessage,
    SystemMessage,
    UserMessage,
    AssistantMessage,
    ChatCompletionsToolDefinition
)

## Step 2: Configure Environment and Azure Connection

In [ ]:
# Load environment variables
load_dotenv()

# Azure AI Project Configuration
project_endpoint = os.getenv("PROJECT_ENDPOINT")
subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")
resource_group_name = os.getenv("AZURE_RESOURCE_GROUP")
project_name = os.getenv("AZURE_AI_PROJECT_NAME")

# Model deployment name
model_deployment_name = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME", "gpt-4o-mini")

# Initialize Azure credential
credential = DefaultAzureCredential()

# Create AI Project client
project_client = AIProjectClient.from_connection_string(
    credential=credential,
    conn_str=project_endpoint
)

print("✅ Connected to Azure AI Foundry")

## Step 3: Create JSON-Based Memory System

We'll implement a simple but effective memory system using JSON files to store user preferences and conversation history.

In [ ]:
class SimpleMemoryStore:
    """Simple JSON-based memory storage for user preferences and history"""
    
    def __init__(self, storage_dir: str = "./agent_memory"):
        self.storage_dir = Path(storage_dir)
        self.storage_dir.mkdir(exist_ok=True)
    
    def _get_user_file(self, user_id: str) -> Path:
        """Get the file path for a user's memory"""
        return self.storage_dir / f"{user_id}_memory.json"
    
    def load_memory(self, user_id: str) -> Dict[str, Any]:
        """Load memory for a specific user"""
        file_path = self._get_user_file(user_id)
        if file_path.exists():
            with open(file_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        return {
            "user_id": user_id,
            "preferences": {},
            "conversation_history": [],
            "last_updated": datetime.now().isoformat()
        }
    
    def save_memory(self, user_id: str, memory_data: Dict[str, Any]):
        """Save memory for a specific user"""
        memory_data["last_updated"] = datetime.now().isoformat()
        file_path = self._get_user_file(user_id)
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(memory_data, f, indent=2, ensure_ascii=False)
    
    def add_preference(self, user_id: str, key: str, value: Any):
        """Add or update a user preference"""
        memory = self.load_memory(user_id)
        memory["preferences"][key] = value
        self.save_memory(user_id, memory)
    
    def get_preference(self, user_id: str, key: str, default=None):
        """Get a user preference"""
        memory = self.load_memory(user_id)
        return memory["preferences"].get(key, default)
    
    def add_conversation_entry(self, user_id: str, role: str, content: str):
        """Add a conversation entry to history"""
        memory = self.load_memory(user_id)
        memory["conversation_history"].append({
            "timestamp": datetime.now().isoformat(),
            "role": role,
            "content": content
        })
        # Keep only last 50 entries to prevent file from growing too large
        memory["conversation_history"] = memory["conversation_history"][-50:]
        self.save_memory(user_id, memory)
    
    def get_conversation_summary(self, user_id: str, last_n: int = 10) -> List[Dict]:
        """Get recent conversation history"""
        memory = self.load_memory(user_id)
        return memory["conversation_history"][-last_n:]
    
    def clear_user_memory(self, user_id: str):
        """Clear all memory for a user"""
        file_path = self._get_user_file(user_id)
        if file_path.exists():
            file_path.unlink()

# Initialize memory store
memory_store = SimpleMemoryStore()
print("✅ Memory store initialized")

## Step 4: Create Memory-Enhanced Agent Tools

These tools allow the agent to remember and recall user information.

In [ ]:
# Global variable to track current user
current_user_id = "user_001"

def save_user_preference(preference_name: str, preference_value: str) -> str:
    """
    Save a user preference to long-term memory.
    
    Args:
        preference_name: The name of the preference (e.g., 'favorite_destination', 'budget_range')
        preference_value: The value of the preference
    
    Returns:
        Confirmation message
    """
    memory_store.add_preference(current_user_id, preference_name, preference_value)
    return f"✅ Saved preference: {preference_name} = {preference_value}"

def get_user_preference(preference_name: str) -> str:
    """
    Retrieve a user preference from long-term memory.
    
    Args:
        preference_name: The name of the preference to retrieve
    
    Returns:
        The preference value or a message if not found
    """
    value = memory_store.get_preference(current_user_id, preference_name)
    if value is None:
        return f"No preference found for '{preference_name}'"
    return f"{preference_name}: {value}"

def get_all_user_preferences() -> str:
    """
    Get all saved user preferences.
    
    Returns:
        JSON string of all preferences
    """
    memory = memory_store.load_memory(current_user_id)
    if not memory["preferences"]:
        return "No preferences saved yet"
    return json.dumps(memory["preferences"], indent=2)

def search_travel_destinations(query: str) -> str:
    """
    Search for travel destinations based on query.
    This is a mock function - in production, this would query a real database.
    
    Args:
        query: Search query for destinations
    
    Returns:
        JSON string of matching destinations
    """
    # Mock travel data
    destinations = [
        {
            "name": "Paris, France",
            "description": "City of lights with world-class museums and cafes",
            "best_season": "Spring/Fall",
            "avg_budget": "$150-300/day",
            "highlights": ["Eiffel Tower", "Louvre Museum", "Cafes"]
        },
        {
            "name": "Tokyo, Japan",
            "description": "Modern metropolis with traditional temples",
            "best_season": "Spring (Cherry Blossoms)",
            "avg_budget": "$120-250/day",
            "highlights": ["Shibuya Crossing", "Temples", "Food Scene"]
        },
        {
            "name": "Bali, Indonesia",
            "description": "Tropical paradise with beaches and culture",
            "best_season": "April-October",
            "avg_budget": "$50-150/day",
            "highlights": ["Beaches", "Rice Terraces", "Temples"]
        },
        {
            "name": "New York, USA",
            "description": "The city that never sleeps",
            "best_season": "Fall",
            "avg_budget": "$200-400/day",
            "highlights": ["Central Park", "Broadway", "Museums"]
        }
    ]
    
    # Simple filtering based on query
    query_lower = query.lower()
    filtered = [d for d in destinations if query_lower in d["name"].lower() or 
                query_lower in d["description"].lower()]
    
    if not filtered:
        filtered = destinations  # Return all if no match
    
    return json.dumps(filtered, indent=2)

print("✅ Agent tools defined")

## Step 5: Create Travel Planning Agent with Memory

Now we'll create an agent that uses the memory tools to provide personalized travel recommendations.

In [ ]:
# Define function definitions for the agent
functions = [
    {
        "type": "function",
        "function": {
            "name": "save_user_preference",
            "description": "Save a user preference to long-term memory (e.g., favorite destinations, budget, travel style)",
            "parameters": {
                "type": "object",
                "properties": {
                    "preference_name": {
                        "type": "string",
                        "description": "Name of the preference (e.g., 'favorite_destination', 'budget_range', 'travel_style')"
                    },
                    "preference_value": {
                        "type": "string",
                        "description": "Value of the preference"
                    }
                },
                "required": ["preference_name", "preference_value"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_user_preference",
            "description": "Retrieve a specific user preference from memory",
            "parameters": {
                "type": "object",
                "properties": {
                    "preference_name": {
                        "type": "string",
                        "description": "Name of the preference to retrieve"
                    }
                },
                "required": ["preference_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_all_user_preferences",
            "description": "Get all saved user preferences at once",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_travel_destinations",
            "description": "Search for travel destinations based on a query",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query for destinations"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

# Create agent with memory-enabled tools
agent = project_client.agents.create_agent(
    model=model_deployment_name,
    name="TravelMemoryAgent",
    instructions="""You are a helpful travel planning assistant with memory capabilities.

Your key abilities:
1. **Remember user preferences**: Always save important user preferences like favorite destinations, budget, travel style, dietary restrictions, etc.
2. **Recall past preferences**: Check saved preferences before making recommendations to personalize suggestions
3. **Learn from conversations**: Pay attention to user statements about likes/dislikes and save them
4. **Provide personalized recommendations**: Use saved preferences to give tailored travel advice

When users mention preferences:
- Proactively save them using save_user_preference
- Use clear preference names like: favorite_destination, budget_range, travel_style, dietary_restrictions, accommodation_preference

When making recommendations:
- First check if there are any relevant saved preferences
- Use those preferences to personalize your suggestions
- Mention that you remember their preferences to build trust

Be conversational, helpful, and always prioritize using memory to improve the user experience.""",
    tools=functions
)

print(f"✅ Created agent: {agent.id}")

# Create a thread for the conversation
thread = project_client.agents.create_thread()
print(f"✅ Created thread: {thread.id}")

## Step 6: Create Helper Function to Run Agent with Tool Calls

In [ ]:
# Map function names to actual functions
function_map = {
    "save_user_preference": save_user_preference,
    "get_user_preference": get_user_preference,
    "get_all_user_preferences": get_all_user_preferences,
    "search_travel_destinations": search_travel_destinations
}

def chat_with_agent(user_message: str, verbose: bool = True):
    """
    Send a message to the agent and handle tool calls.
    
    Args:
        user_message: The user's message
        verbose: Whether to print detailed execution info
    
    Returns:
        The agent's final response
    """
    # Add user message to thread
    project_client.agents.create_message(
        thread_id=thread.id,
        role="user",
        content=user_message
    )
    
    # Also save to our memory store for history
    memory_store.add_conversation_entry(current_user_id, "user", user_message)
    
    if verbose:
        print(f"\n👤 User: {user_message}")
    
    # Create and run
    run = project_client.agents.create_and_process_run(
        thread_id=thread.id,
        agent_id=agent.id
    )
    
    # Handle tool calls if needed
    while run.status == RunStatus.REQUIRES_ACTION:
        if verbose:
            print(f"\n🔧 Agent is calling tools...")
        
        tool_outputs = []
        
        for tool_call in run.required_action.submit_tool_outputs.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            if verbose:
                print(f"  → {function_name}({function_args})")
            
            # Execute the function
            function_to_call = function_map.get(function_name)
            if function_to_call:
                result = function_to_call(**function_args)
                if verbose:
                    print(f"  ✓ Result: {result}")
                
                tool_outputs.append({
                    "tool_call_id": tool_call.id,
                    "output": result
                })
        
        # Submit tool outputs and continue run
        run = project_client.agents.submit_tool_outputs_and_process_run(
            thread_id=thread.id,
            run_id=run.id,
            tool_outputs=tool_outputs
        )
    
    # Get messages
    messages = project_client.agents.list_messages(thread_id=thread.id)
    
    # Get the last assistant message
    assistant_response = None
    for msg in messages:
        if msg.role == "assistant":
            for content in msg.content:
                if hasattr(content, 'text'):
                    assistant_response = content.text.value
                    break
            if assistant_response:
                break
    
    # Save assistant response to memory
    if assistant_response:
        memory_store.add_conversation_entry(current_user_id, "assistant", assistant_response)
    
    if verbose:
        print(f"\n🤖 Assistant: {assistant_response}")
    
    return assistant_response

print("✅ Chat function ready")

## Step 7: Test the Agent - First Conversation (Building Memory)

Let's have an initial conversation where the agent learns about the user's preferences.

In [ ]:
# First interaction - user shares preferences
chat_with_agent("Hi! I'm planning my next vacation. I really love beach destinations and my budget is around $100-200 per day.")

In [ ]:
# Follow-up question
chat_with_agent("What destinations would you recommend?")

## Step 8: View Saved Preferences

Let's check what the agent remembered about the user.

In [ ]:
# Check saved memory
memory = memory_store.load_memory(current_user_id)
print("\n💾 Saved User Memory:")
print(json.dumps(memory["preferences"], indent=2))

## Step 9: Simulate a New Session (Memory Persistence Test)

Now let's create a **new thread** to simulate a completely new conversation session. The agent should still remember the user's preferences from the previous session because they're stored in the JSON file.

In [ ]:
# Create a NEW thread (simulating a new session)
thread = project_client.agents.create_thread()
print(f"✅ Created NEW thread: {thread.id}")
print("   (This simulates the user coming back later)")

# Now ask the agent a question - it should remember preferences from the previous session
chat_with_agent("Hi again! Can you help me plan a trip?")

## Step 10: Test Adding More Preferences

Let's add another preference and see how the agent uses multiple saved preferences together.

In [ ]:
# Add another preference
chat_with_agent("I prefer traveling in the spring or fall to avoid extreme weather.")

In [ ]:
# View all preferences now
print("\n💾 All Saved Preferences:")
print(get_all_user_preferences())

## Step 11: View Conversation History

Let's see the conversation history stored in memory.

In [ ]:
# View recent conversation history
history = memory_store.get_conversation_summary(current_user_id, last_n=5)
print("\n📜 Recent Conversation History:")
for entry in history:
    print(f"\n[{entry['timestamp']}]")
    print(f"{entry['role'].upper()}: {entry['content'][:100]}...")

## Step 12: Multi-User Support Demo

Let's demonstrate that the memory system properly isolates different users.

In [ ]:
# Switch to a different user
current_user_id = "user_002"
print(f"\n🔄 Switched to {current_user_id}")

# Create new thread for new user
thread = project_client.agents.create_thread()
print(f"✅ Created thread for user_002: {thread.id}")

# This user should have no saved preferences
chat_with_agent("Hello! I'm interested in traveling to Japan.")

In [ ]:
# Compare memory for both users
print("\n👥 Memory Comparison:")
print("\n--- User 001 Preferences ---")
user1_memory = memory_store.load_memory("user_001")
print(json.dumps(user1_memory["preferences"], indent=2))

print("\n--- User 002 Preferences ---")
user2_memory = memory_store.load_memory("user_002")
print(json.dumps(user2_memory["preferences"], indent=2))

## Step 13: Cleanup Resources

Clean up the agent and threads when done.

In [ ]:
# Delete the agent (optional - comment out if you want to keep it)
# project_client.agents.delete_agent(agent.id)
# print(f"✅ Deleted agent: {agent.id}")

# Note: Threads and memory files persist
# To delete memory files:
# memory_store.clear_user_memory("user_001")
# memory_store.clear_user_memory("user_002")

print("✅ Demo complete!")

## 🎓 Key Learnings

### What We Built

1. **Simple Memory System**: JSON file-based storage for user preferences and conversation history
2. **Memory-Aware Agent**: Travel agent that saves and recalls user preferences
3. **Multi-User Support**: Separate memory spaces for different users
4. **Persistent Storage**: Preferences survive across different conversation sessions

### Memory Capabilities Demonstrated

- **Short-term Memory**: Conversation context within a session (handled by thread)
- **Long-term Memory**: User preferences persisted across sessions (JSON files)
- **User Isolation**: Each user has their own memory space
- **Memory Retrieval**: Agent actively recalls preferences to personalize responses

### Production Considerations

For production use, consider:
- **Database Storage**: Replace JSON files with a proper database (Azure Cosmos DB, PostgreSQL)
- **Advanced Memory**: Use Mem0 or similar libraries for semantic memory search
- **Vector Storage**: Azure AI Search for semantic similarity searches
- **Memory Expiration**: Implement TTL for stale preferences
- **Privacy & Security**: Encrypt sensitive user data, implement GDPR compliance

### Next Steps

- Explore the Semantic Kernel version (`13-agent-memory.ipynb`) for Mem0 + Azure AI Search integration
- Check out the Cognee version (`13-agent-memory-cognee.ipynb`) for knowledge graph-based memory
- Build on this foundation to create personalized multi-session experiences